In [8]:
# ---------------------------------------------------------
# Cell 1: 匯入所有必要套件
# ---------------------------------------------------------
import pandas as pd
import numpy as np
import warnings
import random
import os
import matplotlib.pyplot as plt
import optuna

from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, StackingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sentence_transformers import SentenceTransformer
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# 補上評估指標套件 (手動防漏 CV 需要)
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, log_loss

random.seed(42)
np.random.seed(42)
warnings.filterwarnings('ignore')

In [9]:
# ---------------------------------------------------------
# Cell 2: 基礎資料清洗、極端值防禦與 Target Encoding
# ---------------------------------------------------------
# 1. 讀取原始資料
train_df = pd.read_csv('../data/boy or girl 2025 train_missingValue.csv')
test_df = pd.read_csv('../data/boy or girl 2025 test no ans_missingValue.csv')

# 2. 清理 phone_os 噪音值
def clean_os(val):
    val = str(val).lower()
    if 'apple' in val or 'ios' in val: return 'Apple'
    if 'android' in val: return 'Android'
    return 'Other'

train_df['phone_os'] = train_df['phone_os'].apply(clean_os)
test_df['phone_os'] = test_df['phone_os'].apply(clean_os)

# 🚀 修正：計算 99 百分位數作為 yt 合理上限
train_df['yt'] = pd.to_numeric(train_df['yt'], errors='coerce')
yt_99_percentile = train_df['yt'].dropna().quantile(0.99)
print(f"yt 欄位 99% 上限設定為: {yt_99_percentile:.0f}")

# 3. 物理邊界強化 (嚴格過濾極端值，防溢位)
def apply_advanced_bounds(df):
    df_clean = df.copy()
    df_clean['yt'] = pd.to_numeric(df_clean['yt'], errors='coerce')
    df_clean['height'] = df_clean['height'].apply(lambda x: x if 100 <= x <= 250 else np.nan)
    df_clean['weight'] = df_clean['weight'].apply(lambda x: x if 30 <= x <= 200 else np.nan)
    df_clean['iq'] = df_clean['iq'].apply(lambda x: x if 50 <= x <= 200 else np.nan)
    df_clean['sleepiness'] = df_clean['sleepiness'].apply(lambda x: x if 0 <= x <= 24 else np.nan)
    df_clean['fb_friends'] = df_clean['fb_friends'].apply(lambda x: x if 0 <= x <= 5000 else np.nan)
    # 🚀 修正：使用 99 百分位數取代原本寬鬆的 1e15
    df_clean['yt'] = df_clean['yt'].apply(lambda x: x if 0 <= x <= yt_99_percentile else np.nan)
    return df_clean

train_df = apply_advanced_bounds(train_df)
test_df = apply_advanced_bounds(test_df)

# 4. 保留類別特徵
X = train_df.drop(columns=['id', 'gender', 'self_intro'])
y = train_df['gender']
X_test_submission = test_df.drop(columns=['id', 'gender', 'self_intro'])
test_ids = test_df['id']

le_y = LabelEncoder()
y_encoded = le_y.fit_transform(y)

# 🚨 修正：為防止 Target Encoding 洩漏，原本的 fit_transform 已移至 Cell 4 的防漏函數內
# print("執行 Target Encoding (Smoothing=20)...")
# te = TargetEncoder(cols=['star_sign', 'phone_os'], smoothing=20)
# X = te.fit_transform(X, y_encoded)
# X_test_submission = te.transform(X_test_submission)
print("📌 Target Encoding 已延後至 CV 內部執行以防止 Data Leakage。")

# 5. 建立缺失值指示器 (保留重要的缺失資訊)
for col in ['weight', 'height', 'yt']:
    X[f'{col}_is_missing'] = X[col].isnull().astype(int)
    X_test_submission[f'{col}_is_missing'] = X_test_submission[col].isnull().astype(int)

print(f"✅ Cell 2 處理完成！目前特徵總數: {X.shape[1]}")

yt 欄位 99% 上限設定為: 99999
📌 Target Encoding 已延後至 CV 內部執行以防止 Data Leakage。
✅ Cell 2 處理完成！目前特徵總數: 11


In [10]:
# ---------------------------------------------------------
# Cell 3: 升級版 NLP 語意萃取 (BGE-Large) 與 PCA 降維
# ---------------------------------------------------------
print("載入 BGE-Large 模型擷取文字語意 (首次執行需下載模型)...")

# 🚀 修正：使用中性字眼取代空字串，並加入 BGE 建議前綴
prefix = "Represent this sentence for searching relevant passages: "
def process_text(text):
    if pd.isna(text) or str(text).strip() == '':
        return 'no description'
    return prefix + str(text)

train_texts = [process_text(t) for t in train_df['self_intro']]
test_texts = [process_text(t) for t in test_df['self_intro']]

# 🚀 升級武器 1：使用 BGE-Large 擷取深層語意
embedder = SentenceTransformer('BAAI/bge-large-en-v1.5')
train_emb = embedder.encode(train_texts, show_progress_bar=True)
test_emb = embedder.encode(test_texts, show_progress_bar=True)

# 🚨 修正：為防止 PCA 全局洩漏，原本的 PCA 運算已移至 Cell 4 的防漏函數內
# print("執行 5 維 Text Embedding PCA...")
# pca = PCA(n_components=5, random_state=42)
# train_pca = pca.fit_transform(train_emb)
# test_pca = pca.transform(test_emb)
print("📌 PCA 降維已延後至 CV 內部執行以防止 Data Leakage。")

print(f"✅ NLP 處理完成！(PCA特徵將在後續動態生成)")

載入 BGE-Large 模型擷取文字語意 (首次執行需下載模型)...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 9408.80it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 14/14 [00:09<00:00,  1.55it/s]

📌 PCA 降維已延後至 CV 內部執行以防止 Data Leakage。
✅ NLP 處理完成！(PCA特徵將在後續動態生成)


In [11]:
# ---------------------------------------------------------
# Cell 4: 隨機森林進階補值與特徵工程
# ---------------------------------------------------------
print("建立進階衍生特徵函數 ( BMI, 體型比例, Log Transform )...")

# 📌 完美保留你的特徵工程邏輯
def create_advanced_features(df):
    df_new = df.copy()
    
    # 1. 建立 BMI 特徵
    df_new['height'] = df_new['height'].clip(lower=1) 
    df_new['BMI'] = df_new['weight'] / ((df_new['height'] / 100) ** 2)
    
    # 🚀 升級武器 2：加入線性的身高體重比 (來自 Willy)
    df_new['height_weight_ratio'] = df_new['height'] / df_new['weight']
    
    # 2. 對極端右偏的數值進行 Log1p 轉換
    df_new['yt_log'] = np.log1p(df_new['yt'].clip(lower=0))
    df_new['fb_friends_log'] = np.log1p(df_new['fb_friends'].clip(lower=0))
    
    # 移除原始尚未 Log 的欄位
    df_new = df_new.drop(columns=['yt', 'fb_friends'])
    return df_new

# 🚨 修正：將剛剛被註解掉的 Target Encoding, PCA, 以及會洩漏的 Imputer 打包成防漏模組
def preprocess_fold(X_tr, y_tr, X_va, tr_emb, va_emb):
    X_tr, X_va = X_tr.copy(), X_va.copy()
    
    # (a) Target Encoding 補回
    te = TargetEncoder(cols=['star_sign', 'phone_os'], smoothing=20)
    X_tr = te.fit_transform(X_tr, y_tr)
    X_va = te.transform(X_va)
    
    # (b) PCA 補回
    pca = PCA(n_components=5, random_state=42)
    tr_pca = pca.fit_transform(tr_emb)
    va_pca = pca.transform(va_emb)
    for i in range(5):
        X_tr[f'intro_pca_{i}'] = tr_pca[:, i]
        X_va[f'intro_pca_{i}'] = va_pca[:, i]
        
    # (c) Random Forest Imputer 補回 (完美保留你原本的設定)
    rf_imputer = IterativeImputer(
        estimator=RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1), 
        random_state=42, max_iter=10
    )
    X_tr_imp = pd.DataFrame(rf_imputer.fit_transform(X_tr), columns=X_tr.columns, index=X_tr.index)
    X_va_imp = pd.DataFrame(rf_imputer.transform(X_va), columns=X_va.columns, index=X_va.index)
    
    # (d) 執行特徵工程
    return create_advanced_features(X_tr_imp), create_advanced_features(X_va_imp)

print("✅ 預處理防漏生產線準備完成！(確保 Imputer 只看 Training Set)")

建立進階衍生特徵函數 ( BMI, 體型比例, Log Transform )...
✅ 預處理防漏生產線準備完成！(確保 Imputer 只看 Training Set)


In [12]:
# ---------------------------------------------------------
# Cell 5: 科學化特徵淘汰 (Permutation Importance)
# ---------------------------------------------------------
print("🚀 升級武器 3：執行 Permutation Importance 自動抓出噪聲特徵...")

# 使用 LightGBM (加上平衡權重) 作為測試模型
lgbm_pi = LGBMClassifier(class_weight='balanced', random_state=42, max_depth=4, verbose=-1)

# 用 5-Fold CV 計算，避免單一切分導致的誤判
cv_pi = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
perm_scores = []
features = None

print("計算重要性中，請稍候...")
for tr_idx, val_idx in cv_pi.split(X, y_encoded):
    # 🚨 修正：在這裡切割原始資料，並呼叫防漏預處理
    X_tr, y_tr = X.iloc[tr_idx], y_encoded[tr_idx]
    X_val, y_val = X.iloc[val_idx], y_encoded[val_idx]
    tr_emb, val_emb = train_emb[tr_idx], train_emb[val_idx]
    
    X_tr_eng, X_val_eng = preprocess_fold(X_tr, y_tr, X_val, tr_emb, val_emb)
    if features is None: features = X_tr_eng.columns
    
    lgbm_pi.fit(X_tr_eng, y_tr)
    
    # 用 ROC-AUC 作為淘汰標準
    result = permutation_importance(
        lgbm_pi, X_val_eng, y_val,
        n_repeats=10, random_state=42, scoring='roc_auc'
    )
    perm_scores.append(result.importances_mean)

perm_importance_mean = np.mean(perm_scores, axis=0)
perm_importance_std = np.std(perm_scores, axis=0)

# 🚀 修正：提高淘汰門檻 (平均值 - 0.5個標準差 > 0)
keep_mask = (perm_importance_mean - 0.5 * perm_importance_std) > 0
removed_features = [features[i] for i in range(len(features)) if not keep_mask[i]]
kept_features = [features[i] for i in range(len(features)) if keep_mask[i]]

print(f"\n🚫 成功移除噪聲特徵 (對 AUC 貢獻不足): {removed_features}")
print(f"✅ 保留核心特徵 ({len(kept_features)} 個): {kept_features}")

🚀 升級武器 3：執行 Permutation Importance 自動抓出噪聲特徵...
計算重要性中，請稍候...

🚫 成功移除噪聲特徵 (對 AUC 貢獻不足): ['phone_os', 'weight_is_missing', 'yt_is_missing', 'intro_pca_0', 'fb_friends_log']
✅ 保留核心特徵 (13 個): ['star_sign', 'height', 'weight', 'sleepiness', 'iq', 'height_is_missing', 'intro_pca_1', 'intro_pca_2', 'intro_pca_3', 'intro_pca_4', 'BMI', 'height_weight_ratio', 'yt_log']


In [13]:
# ---------------------------------------------------------
# Cell 6: Optuna 最佳化與終極 Stacking 驗證
# ---------------------------------------------------------
print("啟動 Optuna 自動尋找最強參數 (目標: 最大化 ROC-AUC)...")

ratio = (y_encoded == 0).sum() / (y_encoded == 1).sum()

def objective(trial):
    # 📌 完美保留你設定的 XGBoost, LightGBM, Random Forest 參數區間
    xgb_params = {
        'scale_pos_weight': ratio, 'eval_metric': 'logloss', 'random_state': 42,
        'tree_method': 'hist', 'device': 'cuda', 
        'max_depth': trial.suggest_int('xgb_max_depth', 3, 7),
        'learning_rate': trial.suggest_float('xgb_lr', 0.01, 0.1, log=True),
        'n_estimators': trial.suggest_int('xgb_n', 100, 300),
        'subsample': trial.suggest_float('xgb_sub', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('xgb_col', 0.6, 1.0),
        'reg_lambda': trial.suggest_float('xgb_lambda', 1e-3, 10.0, log=True)
    }
    
    lgbm_params = {
        'class_weight': 'balanced', 'random_state': 42, 'verbose': -1,
        'max_depth': trial.suggest_int('lgbm_max_depth', 3, 7),
        'num_leaves': trial.suggest_int('lgbm_leaves', 15, 63),
        'learning_rate': trial.suggest_float('lgbm_lr', 0.01, 0.1, log=True),
        'n_estimators': trial.suggest_int('lgbm_n', 100, 300)
    }
    
    rf_params = {
        'class_weight': 'balanced', 'random_state': 42,
        'max_depth': trial.suggest_int('rf_max_depth', 4, 10),
        'n_estimators': trial.suggest_int('rf_n', 100, 300),
        'min_samples_leaf': trial.suggest_int('rf_min_leaf', 1, 5)
    }
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) 
    scores = []
    
    for tr_idx, val_idx in skf.split(X, y_encoded):
        X_tr, y_tr = X.iloc[tr_idx], y_encoded[tr_idx]
        X_va, y_va = X.iloc[val_idx], y_encoded[val_idx]
        tr_emb, va_emb = train_emb[tr_idx], train_emb[val_idx]
        
        # 🚨 使用防漏處理，確保不洩漏
        X_tr_eng, X_va_eng = preprocess_fold(X_tr, y_tr, X_va, tr_emb, va_emb)
        X_tr_pruned, X_va_pruned = X_tr_eng[kept_features], X_va_eng[kept_features]
        
        model = StackingClassifier(
            estimators=[('XGB', XGBClassifier(**xgb_params)), ('LGBM', LGBMClassifier(**lgbm_params)), ('RF', RandomForestClassifier(**rf_params))],
            final_estimator=LogisticRegression(class_weight='balanced', random_state=42)
        )
        model.fit(X_tr_pruned, y_tr)
        preds = model.predict_proba(X_va_pruned)[:, 1]
        scores.append(roc_auc_score(y_va, preds))
        
    return np.mean(scores)

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20) 

print("\n🏆 Optuna 搜尋完成！找到最佳參數組合：")
best_p = study.best_params

print("\n使用最佳參數建立最終的 Stacking 模型...")
# 📌 完美還原你的 Stacking 模型組裝
final_xgb = XGBClassifier(
    scale_pos_weight=ratio, eval_metric='logloss', random_state=42, tree_method='hist', device='cuda',
    max_depth=best_p['xgb_max_depth'], learning_rate=best_p['xgb_lr'], n_estimators=best_p['xgb_n'],
    subsample=best_p['xgb_sub'], colsample_bytree=best_p['xgb_col'], reg_lambda=best_p['xgb_lambda']
)

final_lgbm = LGBMClassifier(
    class_weight='balanced', random_state=42, verbose=-1,
    max_depth=best_p['lgbm_max_depth'], num_leaves=best_p['lgbm_leaves'],
    learning_rate=best_p['lgbm_lr'], n_estimators=best_p['lgbm_n']
)

final_rf = RandomForestClassifier(
    class_weight='balanced', random_state=42,
    max_depth=best_p['rf_max_depth'], n_estimators=best_p['rf_n'], min_samples_leaf=best_p['rf_min_leaf']
)

stacking_model = StackingClassifier(
    estimators=[('XGB', final_xgb), ('LGBM', final_lgbm), ('RF', final_rf)],
    final_estimator=LogisticRegression(class_weight='balanced', random_state=42)
)

print("進行 Stratified 10-Fold 交叉驗證中 (評估終極版 Stacking 模型)...")
skf_10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# 🚨 修正：為了保留你的報表又不能洩漏，這裡使用手動迴圈產出完全相同的指標格式
acc_list, auc_list, f1_list, logloss_list = [], [], [], []

for tr_idx, val_idx in skf_10.split(X, y_encoded):
    X_tr, y_tr = X.iloc[tr_idx], y_encoded[tr_idx]
    X_va, y_va = X.iloc[val_idx], y_encoded[val_idx]
    tr_emb, va_emb = train_emb[tr_idx], train_emb[val_idx]
    
    X_tr_eng, X_va_eng = preprocess_fold(X_tr, y_tr, X_va, tr_emb, va_emb)
    X_tr_pruned, X_va_pruned = X_tr_eng[kept_features], X_va_eng[kept_features]
    
    stacking_model.fit(X_tr_pruned, y_tr)
    preds_prob = stacking_model.predict_proba(X_va_pruned)
    preds_class = stacking_model.predict(X_va_pruned)
    
    acc_list.append(accuracy_score(y_va, preds_class))
    auc_list.append(roc_auc_score(y_va, preds_prob[:, 1]))
    f1_list.append(f1_score(y_va, preds_class))
    logloss_list.append(log_loss(y_va, preds_prob))

# 📌 完美保留你的報表格式
print("\n📊 === Ultimate 終極版 Stacking 10-Fold CV 綜合評估報告 ===")
print(f"Accuracy (準確率) : {np.mean(acc_list):.4f} (std: {np.std(acc_list):.4f})")
print(f"ROC-AUC  (排序力) : {np.mean(auc_list):.4f} (std: {np.std(auc_list):.4f})")
print(f"F1-Score (平衡度) : {np.mean(f1_list):.4f} (std: {np.std(f1_list):.4f})")
print(f"Log Loss (誤差值) : {np.mean(logloss_list):.4f} (std: {np.std(logloss_list):.4f})")

[I 2026-03-26 10:30:14,478] A new study created in memory with name: no-name-f594a0ea-f367-4c1e-9b4f-b2e0bc929d5b


啟動 Optuna 自動尋找最強參數 (目標: 最大化 ROC-AUC)...


[I 2026-03-26 10:31:34,722] Trial 0 finished with value: 0.9158581993403422 and parameters: {'xgb_max_depth': 4, 'xgb_lr': 0.027252285107576272, 'xgb_n': 215, 'xgb_sub': 0.7362554338794969, 'xgb_col': 0.6497296623098214, 'xgb_lambda': 0.012951718661362594, 'lgbm_max_depth': 5, 'lgbm_leaves': 43, 'lgbm_lr': 0.019402383451064653, 'lgbm_n': 190, 'rf_max_depth': 6, 'rf_n': 250, 'rf_min_leaf': 2}. Best is trial 0 with value: 0.9158581993403422.
[I 2026-03-26 10:32:51,802] Trial 1 finished with value: 0.9175292465471037 and parameters: {'xgb_max_depth': 5, 'xgb_lr': 0.011389710823364953, 'xgb_n': 194, 'xgb_sub': 0.9074307240249565, 'xgb_col': 0.7078520221371312, 'xgb_lambda': 0.021148838896105155, 'lgbm_max_depth': 5, 'lgbm_leaves': 26, 'lgbm_lr': 0.048976278110304486, 'lgbm_n': 199, 'rf_max_depth': 4, 'rf_n': 162, 'rf_min_leaf': 5}. Best is trial 1 with value: 0.9175292465471037.
[I 2026-03-26 10:34:18,701] Trial 2 finished with value: 0.9177744708994708 and parameters: {'xgb_max_depth': 6,


🏆 Optuna 搜尋完成！找到最佳參數組合：

使用最佳參數建立最終的 Stacking 模型...
進行 Stratified 10-Fold 交叉驗證中 (評估終極版 Stacking 模型)...

📊 === Ultimate 終極版 Stacking 10-Fold CV 綜合評估報告 ===
Accuracy (準確率) : 0.8841 (std: 0.0607)
ROC-AUC  (排序力) : 0.9299 (std: 0.0668)
F1-Score (平衡度) : 0.7900 (std: 0.1078)
Log Loss (誤差值) : 0.3803 (std: 0.1129)


In [14]:
# ---------------------------------------------------------
# Cell 7: 終極預測與存檔 (Ultimate 版)
# ---------------------------------------------------------
# 🚨 修正：預測時使用全部 Train Data 進行防漏前處理轉換
print("正在將全局資料通過防漏生產線進行轉換...")
X_train_final, X_test_final = preprocess_fold(X, y_encoded, X_test_submission, train_emb, test_emb)

X_pruned = X_train_final[kept_features]
X_test_pruned = X_test_final[kept_features]

# 📌 完美保留你的所有 Print 與存檔路徑
print(f"目前訓練集特徵數: {X_pruned.shape[1]}") 
print(f"目前測試集特徵數: {X_test_pruned.shape[1]}") 

# 用所有資料正式訓練終極版模型
print("正式訓練 Ultimate 終極版 Stacking 模型中...")
stacking_model.fit(X_pruned, y_encoded)
print("訓練完成！")

print("進行測試集預測中...")
test_predictions_encoded = stacking_model.predict(X_test_pruned) 

# 將 0, 1 轉回原始的性別標籤
test_predictions_original = le_y.inverse_transform(test_predictions_encoded)

submission_df = pd.DataFrame({
    'id': test_ids, 
    'gender': test_predictions_original
})

output_dir = '../results'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

file_path = os.path.join(output_dir, 'submission_ultimate.csv')
submission_df.to_csv(file_path, index=False)

print(f"🏆 終極版預測成功！檔案已儲存至: {file_path}")
print("前五筆預測結果：")
print(submission_df.head())

正在將全局資料通過防漏生產線進行轉換...
目前訓練集特徵數: 13
目前測試集特徵數: 13
正式訓練 Ultimate 終極版 Stacking 模型中...
訓練完成！
進行測試集預測中...
🏆 終極版預測成功！檔案已儲存至: ../results\submission_ultimate.csv
前五筆預測結果：
   id  gender
0   1       1
1   2       1
2   3       2
3   4       1
4   5       2
